# Cold-Start Benchmark Development and Final Experiment

This notebook documents the design, implementation, execution, quality assurance and analysis of the JavaScript runtime cold-start benchmark.

The benchmark compares the startup performance of:

- Node.js;
- Bun; and
- Deno.

Cold-start time is one of the dependent variables investigated in the study.

## 1. Purpose of the Cold-Start Benchmark

The cold-start benchmark measures the time required for each JavaScript runtime to launch a new process, initialise a small JavaScript program and signal that it is ready.

Cold-start time is distinct from HTTP latency. HTTP latency measures the time required to process a request after a server is already running. Cold-start time measures the overhead involved in creating and initialising a new runtime process.

This metric is relevant to short-lived scripts, command-line tools, serverless functions and dynamically scaled services in which JavaScript processes may be started frequently.

## 2. Cold-Start Workload

A small startup program was created separately for Node.js, Bun and Deno.

Each program performed the same operations:

1. create a small JavaScript object;
2. serialise the object using `JSON.stringify()`; and
3. write the exact string `READY` to standard output.

The workload did not include:

- file access;
- network communication;
- database operations;
- third-party packages; or
- asynchronous waiting.

A new operating-system process was created for every observation. Runtime processes were not reused between measurements.## 2. Cold-Start Workload

A small startup program was created separately for Node.js, Bun and Deno.

Each program performed the same operations:

1. create a small JavaScript object;
2. serialise the object using `JSON.stringify()`; and
3. write the exact string `READY` to standard output.

The workload did not include:

- file access;
- network communication;
- database operations;
- third-party packages; or
- asynchronous waiting.

A new operating-system process was created for every observation. Runtime processes were not reused between measurements.

## 3. Measurement Definition

Cold-start time was defined as the elapsed monotonic time from immediately before creation of the runtime process until the exact `READY` signal was received from the program's standard output.

The Python `time.perf_counter_ns()` function was used to obtain high-resolution monotonic timestamps before process creation and after reception of the readiness signal.

The measured duration therefore included:

- operating-system process creation;
- runtime executable loading;
- JavaScript runtime initialisation;
- startup-program loading and parsing;
- execution of the small startup workload; and
- transmission of the readiness signal through standard output.

Process shutdown after the readiness signal was received was not included in the measured value.

## 4. Final Experimental Protocol

| Experimental property | Final configuration |
|---|---|
| Independent variable | JavaScript runtime |
| Runtime levels | Node.js, Bun and Deno |
| Dependent variable | Cold-start time |
| Primary unit | Milliseconds |
| Repetitions | 30 per runtime |
| Collection sessions | 2 |
| Repetitions per session | 15 |
| Observations per session | 45 |
| Total observations | 90 |
| Process reuse | No |
| New process per observation | Yes |
| Readiness signal | `READY` |
| Timing source | High-resolution monotonic timer |
| Runtime order | Randomised within every repetition |
| Cool-down interval | 1 second |
| Observation timeout | 10 seconds |

The execution order of Node.js, Bun and Deno was randomised independently within every repetition using a fixed random seed. This prevented one runtime from consistently being executed first or last.

The experiment was divided into two balanced sessions. Session one contained repetitions 1–15, while session two contained repetitions 16–30.

In [1]:
library(dplyr)

candidate_manifest_paths <- c(
  "data/raw/cold_start/cold_start_final_manifest.csv",
  "../data/raw/cold_start/cold_start_final_manifest.csv"
)

available_manifest_paths <- candidate_manifest_paths[
  file.exists(candidate_manifest_paths)
]

if (length(available_manifest_paths) == 0) {
  stop(
    paste(
      "The cold-start manifest could not be found.",
      "Check the notebook working directory."
    )
  )
}

cold_start_manifest_path <- normalizePath(
  available_manifest_paths[1],
  winslash = "/",
  mustWork = TRUE
)

cold_start_final <- read.csv(
  cold_start_manifest_path,
  stringsAsFactors = FALSE
)

cold_start_final <- cold_start_final |>
  mutate(
    session_id = as.integer(session_id),
    repetition = as.integer(repetition),
    startup_time_ns = as.numeric(startup_time_ns),
    startup_time_ms = as.numeric(startup_time_ms)
  )

cat(
  "Manifest path:\n",
  cold_start_manifest_path,
  "\n\n"
)

cat(
  "Rows:",
  nrow(cold_start_final),
  "\n"
)

cat(
  "Columns:",
  ncol(cold_start_final),
  "\n"
)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




Manifest path:
 D:/FOLO_PROJECTS/MASTERS/javascript-runtime-performance-study/data/raw/cold_start/cold_start_final_manifest.csv 

Rows: 90 
Columns: 16 


In [2]:
# number of obervations per runtime

cold_start_runtime_counts <- cold_start_final |>
  count(
    runtime,
    name = "observations"
  ) |>
  arrange(runtime)

cold_start_runtime_counts

runtime,observations
<chr>,<int>
bun,30
deno,30
node,30


In [3]:
cold_start_session_counts <- cold_start_final |>
  count(
    session_id,
    runtime,
    name = "observations"
  ) |>
  arrange(
    session_id,
    runtime
  )

cold_start_session_counts

session_id,runtime,observations
<int>,<chr>,<int>
1,bun,15
1,deno,15
1,node,15
2,bun,15
2,deno,15
2,node,15


In [4]:
# check for duplicates
cold_start_duplicate_check <- cold_start_final |>
  count(
    runtime,
    repetition,
    name = "occurrences"
  ) |>
  filter(
    occurrences != 1
  )

cold_start_duplicate_check

runtime,repetition,occurrences
<chr>,<int>,<int>


In [5]:
#check for missing observations

expected_cold_start_design <- expand.grid(
  runtime = c(
    "node",
    "bun",
    "deno"
  ),
  repetition = 1:30,
  stringsAsFactors = FALSE
)

missing_cold_start_observations <- expected_cold_start_design |>
  anti_join(
    cold_start_final |>
      select(
        runtime,
        repetition
      ),
    by = c(
      "runtime",
      "repetition"
    )
  )

missing_cold_start_observations

runtime,repetition
<chr>,<int>


In [6]:
# check for completion status

cold_start_status_summary <- cold_start_final |>
  count(
    status,
    name = "observations"
  )

cold_start_status_summary

status,observations
<chr>,<int>
success,90


In [7]:
cold_start_audit_summary <- data.frame(
  check = c(
    "Total observations",
    "Unique run identifiers",
    "Runtime groups",
    "Runtime groups with 30 observations",
    "Session-runtime groups",
    "Session-runtime groups with 15 observations",
    "Duplicate runtime-repetition combinations",
    "Missing runtime-repetition combinations",
    "Failed observations",
    "Unexpected readiness signals",
    "Missing startup times",
    "Non-positive startup times"
  ),

  result = c(
    nrow(cold_start_final),

    n_distinct(
      cold_start_final$run_id
    ),

    nrow(
      cold_start_runtime_counts
    ),

    sum(
      cold_start_runtime_counts$observations == 30
    ),

    nrow(
      cold_start_session_counts
    ),

    sum(
      cold_start_session_counts$observations == 15
    ),

    nrow(
      cold_start_duplicate_check
    ),

    nrow(
      missing_cold_start_observations
    ),

    sum(
      cold_start_final$status != "success"
    ),

    sum(
      cold_start_final$observed_output != "READY"
    ),

    sum(
      is.na(cold_start_final$startup_time_ms)
    ),

    sum(
      cold_start_final$startup_time_ms <= 0,
      na.rm = TRUE
    )
  ),

  expected = c(
    90,
    90,
    3,
    3,
    6,
    6,
    0,
    0,
    0,
    0,
    0,
    0
  )
) |>
  mutate(
    passed = result == expected
  )

cold_start_audit_summary

check,result,expected,passed
<chr>,<int>,<dbl>,<lgl>
Total observations,90,90,TRUE
Unique run identifiers,90,90,TRUE
Runtime groups,3,3,TRUE
Runtime groups with 30 observations,3,3,TRUE
Session-runtime groups,6,6,TRUE
Session-runtime groups with 15 observations,6,6,TRUE
Duplicate runtime-repetition combinations,0,0,TRUE
Missing runtime-repetition combinations,0,0,TRUE
Failed observations,0,0,TRUE


In [9]:
# save the dataset
project_root <- if (
  dir.exists("data")
) {
  "."
} else {
  ".."
}

cold_start_processed_directory <- file.path(
  project_root,
  "data",
  "processed",
  "cold_start"
)

cold_start_table_directory <- file.path(
  project_root,
  "results",
  "tables",
  "cold_start"
)

dir.create(
  cold_start_processed_directory,
  recursive = TRUE,
  showWarnings = FALSE
)

dir.create(
  cold_start_table_directory,
  recursive = TRUE,
  showWarnings = FALSE
)

cold_start_processed_path <- file.path(
  cold_start_processed_directory,
  "cold_start_final.csv"
)

write.csv(
  cold_start_final,
  cold_start_processed_path,
  row.names = FALSE
)

write.csv(
  cold_start_audit_summary,
  file.path(
    cold_start_table_directory,
    "cold_start_final_audit_summary.csv"
  ),
  row.names = FALSE
)

cat(
  "Processed dataset:\n",
  normalizePath(
    cold_start_processed_path,
    winslash = "/",
    mustWork = TRUE
  )
)

Processed dataset:
 D:/FOLO_PROJECTS/MASTERS/javascript-runtime-performance-study/data/processed/cold_start/cold_start_final.csv

## 5. Final Data Collection and Quality Assurance

The final cold-start experiment produced 90 observations. Each of the three JavaScript runtimes was measured 30 times.

The observations were divided across two balanced collection sessions:

- session one contained repetitions 1–15; and
- session two contained repetitions 16–30.

Within each repetition, the execution order of Node.js, Bun and Deno was randomised. A new operating-system process was created for every observation.

A post-collection quality audit confirmed that:

- all 90 planned observations were present;
- all run identifiers were unique;
- each runtime contained exactly 30 observations;
- each runtime contained 15 observations in each session;
- no runtime-and-repetition combinations were duplicated;
- no planned combinations were missing;
- all observations completed successfully;
- every program returned the expected `READY` signal;
- no startup-time measurements were missing; and
- all recorded startup times were greater than zero.

The audited dataset was saved as:

`data/processed/cold_start/cold_start_final.csv`

The raw observation files, execution plans, logs, manifest and frozen experimental metadata were retained.

## 6. Deferred Analysis

Descriptive statistics, visualisations, inferential statistical tests and interpretation of the cold-start results were intentionally deferred until data collection and quality assurance had been completed for all dependent variables in the study.